# PDF → Markdown Parser & Chunker — Test Notebook

### Workflow
```
.env  →  load_config_from_env()  →  PDFToMarkdown.convert()  →  MarkdownChunker.chunk()
```
| Step | Cell | What it does |
|---|---|---|
| 0 | Install | pip install required packages |
| 1 | Config  | Load Azure OpenAI credentials from `.env` |
| 2 | Heuristic parse | Convert PDF without LLM (fast baseline) |
| 3 | Integrity check | Review content-retention stats |
| 4 | LLM parse | Full Azure OpenAI parse |
| 5 | Hybrid parse | Heuristic + LLM refinement (best quality) |
| 6 | Inspect headings | Validate heading hierarchy |
| 7 | Chunk | Split Markdown into RAG-ready chunks |
| 8 | Validate chunks | Size, page metadata, heading outline |
| 9 | Export | Save `chunks.json` |

In [ ]:
# ── STEP 0: Install packages ──────────────────────────────────────────────────
# Run once, then restart the kernel.
# !pip install pdfplumber pypdf openai -q

In [ ]:
import importlib, re, json, statistics
from pathlib import Path
from collections import Counter
from IPython.display import display, Markdown

import pdf_to_markdown as ptm
importlib.reload(ptm)

from pdf_to_markdown import (
    ParserConfig,
    HeuristicPDFParser,
    LLMPDFParser,
    PDFToMarkdown,
    MarkdownChunker,
    Chunk,
    ParseStats,
)

print("Imports OK — module reloaded")


Imports OK — module reloaded


## Step 1 — Configure Parser

Set the vLLM endpoint URL and model name in `ParserConfig`.  
Default values point to a local vLLM server (`http://localhost:8000/v1`, model `mistral-small-24b`).


In [ ]:
# ── STEP 1: Configure ParserConfig ──────────────────────────────────────────
# ParserConfig defaults point to a local vLLM endpoint.
# Override as needed:
#   llm_base_url  : URL of the vLLM OpenAI-compatible endpoint
#   llm_model     : model name served by vLLM
#   llm_max_tokens, llm_page_batch, header_zone, … (see ParserConfig for all options)

cfg = ParserConfig(
    # llm_base_url="http://localhost:8000/v1",   # default
    # llm_model="mistral-small-24b",            # default
    # llm_max_tokens=4096,
    # llm_page_batch=5,
    # header_zone=0.07,
)

# ── Path to the PDF you want to test ────────────────────────────────────────
PDF_PATH = "Anlage_2_Checkliste_Freelancer_Selbständige .pdf"   # ← change to your PDF file

print(f"\nPDF path   : {PDF_PATH}  (exists={Path(PDF_PATH).exists()})")
print(f"llm_base_url : {cfg.llm_base_url}")
print(f"llm_model    : {cfg.llm_model}")


[load_config_from_env] Backend  : Azure OpenAI
[load_config_from_env] Endpoint : https://ba-account-openai-france.openai.azure.com
[load_config_from_env] Deploy   : gpt-4o
[load_config_from_env] API key  : set

PDF path   : Anlage_2_Checkliste_Freelancer_Selbständige .pdf  (exists=True)
use_azure  : True
deployment : gpt-4o


## Step 2 — Heuristic Parse (fast, no LLM)

Baseline conversion using only font-size and layout signals.  
Run this first to see how well the document can be parsed without AI.

In [33]:
# ── STEP 2: Heuristic parse ──────────────────────────────────────────────────
converter = PDFToMarkdown(cfg)
md_heuristic = converter.convert(PDF_PATH, output_path="output_heuristic.md", strategy="heuristic")

print(f"\nOutput: {len(md_heuristic):,} characters → output_heuristic.md")


[PDF->MD] Input    : Anlage_2_Checkliste_Freelancer_Selbständige .pdf
[PDF->MD] Strategy : heuristic
  Body text size  : 11.0 pt
  Heading sizes   : [14.0]
  HF patterns     : 1 recurring pattern(s) removed
  Content retention : 86.2%  (7,075 / 8,204 body chars)
[PDF->MD] Saved -> output_heuristic.md

Output: 8,290 characters → output_heuristic.md


In [34]:
# ── STEP 3: Content-integrity report ─────────────────────────────────────────
stats: ParseStats | None = converter._heuristic.last_parse_stats

if stats is None:
    print("No stats available.")
else:
    bar = "─" * 55
    print(bar)
    print(f"  Total blocks extracted    : {stats.total_blocks}")
    print(f"  Empty groups dropped      : {stats.empty_blocks_dropped}  (no extractable chars)")
    print(f"  HF blocks removed         : {stats.hf_blocks_dropped}  (header/footer detection)")
    print(f"  Body chars input          : {stats.chars_body_input:,}")
    print(f"  Markdown chars output     : {stats.chars_output:,}")
    print(f"  Content retention         : {stats.content_retention_pct:.1f}%")
    print(bar)

    if stats.warnings:
        print()
        for w in stats.warnings:
            print(f"  ⚠  {w}")

    if stats.hf_dropped_samples:
        print(f"\nHeader/footer blocks removed ({len(stats.hf_dropped_samples)} shown):")
        print("Review — confirm none are real content paragraphs.\n")
        for pg, text in stats.hf_dropped_samples:
            suffix = "…" if len(text) >= 100 else ""
            print(f"  [p{pg:>3}]  {text[:100]}{suffix}")

───────────────────────────────────────────────────────
  Total blocks extracted    : 24
  Empty groups dropped      : 0  (no extractable chars)
  HF blocks removed         : 4  (header/footer detection)
  Body chars input          : 8,204
  Markdown chars output     : 7,075
  Content retention         : 86.2%
───────────────────────────────────────────────────────

Header/footer blocks removed (4 shown):
Review — confirm none are real content paragraphs.

  [p  1]  181019_Checkliste_Scheinselbständigkeit
  [p  2]  181019_Checkliste_Scheinselbständigkeit
  [p  3]  181019_Checkliste_Scheinselbständigkeit
  [p  4]  181019_Checkliste_Scheinselbständigkeit


In [35]:
# ── Preview rendered Markdown ────────────────────────────────────────────────
PREVIEW_CHARS = 6000
preview = md_heuristic[:PREVIEW_CHARS]
if len(md_heuristic) > PREVIEW_CHARS:
    preview += f"\n\n*… [{len(md_heuristic) - PREVIEW_CHARS:,} more characters]*"

display(Markdown(preview))

<!--page:1-->

# Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-

Für das IT-Systemhaus werden im Rahmen von Dienstleistungs- und Werkverträgen unter anderem Freelancer1 tätig. Der Einsatz erfolgt entweder durch Abrufe aus dem Vertrag „Free- lancer“ oder durch Abrufe von Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer. Es kann in der täglichen Arbeit zu Situationen kommen, die als (Teil-) Indizien für das Vorliegen von Scheinselbständigkeit angesehen werden könnten, diese sollten vermieden werden. Solche Indizien liegen insbesondere dann vor, wenn der Verdacht besteht, dass Freelancer in den Arbeitsablauf der BA-IT integriert sind. Die Wertung, ob eine Selbstän- digkeit oder eine Arbeitnehmereigenschaft vorliegt, richtet sich in jedem Einzelfall nach den Gesamtumständen.
Nachstehende Checkliste dient dem Abgleich, ob in Ihrem Zuständigkeitsbereich in der Zu- sammenarbeit mit Freelancern Sachverhalte bestehen, die möglicherweise auf das Vorliegen einer Scheinselbständigkeit schließen lassen könnten. Bitte füllen Sie die Liste persönlich unter Beteiligung des Freelancers aus und stimmen Sie das Ergebnis mit Ihrer/Ihrem Vorge- setzten ab. Lfd. Nummern, die mit * gekennzeichnet sind betreffen den Sachverhalt „Freelan- cern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer“.

| Organisationseinheit/ Projekt |
|---|
| Vertragsname |
| Abruf – Nr. |
| Name, Vorname (Freelancer) |
| Name, Vorname (Abrufverantwortliche/r IT-Systemhaus) |
| Name, Vorname (zust. GBL IT-Systemhaus) |

| Lfd. Nr. | Sachverhalt | ja | nein | Bemerkungen | Wichtigkeit / Indizwir- kung für Scheinselb- ständigkeit |
|---|---|---|---|---|---|
| 1 * | Für die Kommunikation mit dem Freelancer stehen namentlich benannte Ansprechpart- ner des IT-Dienstleistungsvermittlers bzw. externen Dienstleistungsunternehmens (im Folgenden: Auftragnehmer) zur Verfügung. |  |  |  | sehr hoch |
| 2 | Einsatzzeit und Einsatzumfang des Free- lancers wird so gestaltet, dass der Freelan- |  |  |  | sehr hoch |

1 Freelancer = Selbständiger, der aufgrund eines Dienst- oder Werkvertragsverhältnisses Aufträge für den Auf- traggeber durchführt.
<!--page:2-->

| Lfd. Nr. | Sachverhalt | ja | nein | Bemerkungen | Wichtigkeit / Indizwir- kung für Scheinselb- ständigkeit |
|---|---|---|---|---|---|
|  | cer Aufträge anderer Auftraggeber bearbei- ten kann. Der Freelancer ist nicht aus- schließlich und auf Dauer für die BA tätig. |  |  |  |  |
| 3 | Gemeinsame Einsatzpläne werden grund- sätzlich nicht aufgestellt, es sei denn, es ist zur Aufgabenerledigung erforderlich und der gemeinsame Einsatzplan ist mit dem namentlich benannten Ansprechpartner des Auftragnehmers abgestimmt. Die ggfs. be- stehende Erforderlichkeit ist jeweils begrün- det und dokumentiert. |  |  |  | sehr hoch |
| 4 | Es besteht grundsätzlich keine Anwesen- heitspflicht für Freelancer. |  |  |  | sehr hoch |
| 5 | Die konkrete Auftragsdefinition und das er- wartete Ergebnis werden im Abruf durch die BA definiert und in weiteren konkretisieren- den Arbeitsaufträgen beschrieben. |  |  |  | sehr hoch |
| 6 | Freelancer nehmen nicht an Dienstbespre- chungen teil, es sei denn, dies ist aus fach- lichen Gründen zu bestimmten Aspekten o- der Tagesordnungspunkten aus Gründen der Auftragserledigung sinnvoll und erfor- derlich. Die ggfs. bestehende Erforderlichkeit ist im Protokoll jeweils begründet und dokumen- tiert. Die Teilnehmer werden vor Beginn der Dienstbesprechung über die Anwesenheit Externer unterrichtet. |  |  |  | hoch |
| 7 | Freelancer und interne Mitarbeiter sind räumlich getrennt untergebracht, es sei denn die Aufgabenerledigung erfordert die gemeinsame Bearbeitung. |  |  |  | gering |
| 8 | Türschilder der Büroräume, in denen Free- lancer arbeiten, sind entsprechend beschrif- tet. |  |  |  | hoch |
| 9 | Freelancer besitzen keine BA-Visitenkarten. |  |  |  | hoch |
| 10 | Freelancer erhalten PKI-Gastkarten und sind nicht mit „normalen“ Dienstkarten aus- gestattet. |  |  |  | hoch |
| 11 | Freelancer verfügen nicht über einen BA- spezifischen E-Mail-Account. |  |  |  | hoch |

<!--page:3-->

| Lfd. Nr. | Sachverhalt | ja | nein | Bemerkungen | Wichtigkeit / Indizwir- kung für Scheinselb- ständigkeit |
|---|---|---|---|---|---|
| 12 | Freelancer sind in funktionsbezogenen E- Mail-Verteilern der BA nicht enthalten. |  |  |  | hoch |
| 13 | Freelancer sind in Telefon-, Namens- oder ähnlichen Verzeichnissen eindeutig als Ex- terne gekennzeichnet. Sie weisen sich in ihrer Kommunikation ausdrücklich als Externe aus (z.B. E-Mail- Signatur) und tragen im Dienstgebäude der BA ein Namensschild mit Firmennamen des Auftragnehmers bzw. der Kennzeichnung als extern. |  |  |  | hoch |
| 14 | IT-Arbeitsmittel und die Netzinfrastruktur der BA werden nur insoweit zur Verfügung gestellt, als dies zur Auftragserledigung er- forderlich ist |  |  |  | hoch |
| 15 | Ein Zugriff auf BA-spezifische Ablagen, Da- teien und auf das Intranet findet nicht statt, es sei denn dies ist zur Auftragserledigung erforderlich. |  |  |  | hoch |
| 16 * | Mit dem Auftragnehmer hat eine Kick-Off- Veranstaltung zur Zusammenarbeit stattge- funden. Regelungen zur Zusammenarbeit wurden besprochen. Das Thema wird regelmäßig, mindestens einmal jährlich (wiederholend) besprochen. |  |  |  | gering |
| 17 * | Freelancer werden ausschließlich über den Auftragnehmer (z.B. durch den namentlich benannten Ansprechpartner) in Bezug auf Gefahren für Sicherheit und Gesundheit un- terwiesen. Hinweis: Für die Unterweisung zum Arbeitsschutz- und Sicherheit der Freelancer ist der Auf- tragnehmer zuständig. Das IT-Systemhaus stellt dem namentlich benannten Ansprech- partner die relevanten Informationen zur Verfügung. Arbeitsschutz/Arbeitssicherheit |  |  |  | gering |

Das gemeinsame Gespräch fand statt am:
Die in der Checkliste vorgenommenen Eintragungen werden hiermit bestätigt:
<!--page:4-->
Datum, Unterschrift (Abrufverantwortlicher IT-Systemhaus)
Ich war 

*… [2,290 more characters]*

## Step 4 — LLM Parse (vLLM / Mistral Small 24B)

Sends raw page text to the vLLM backend. Best for documents where font-size signals are unreliable (e.g. no formatted headings, scanned, or complex layouts).


In [ ]:
# ── STEP 4: Full LLM parse via vLLM ─────────────────────────────────────────
# Requires a running vLLM server at cfg.llm_base_url.

md_llm = converter.convert(PDF_PATH, output_path="output_llm.md", strategy="llm")

print(f"\nOutput: {len(md_llm):,} characters → output_llm.md")
display(Markdown(md_llm[:4000]))



[PDF->MD] Input    : Anlage_2_Checkliste_Freelancer_Selbständige .pdf
[PDF->MD] Strategy : llm
  LLM parsing pages 1–4 / 4 (7,981 chars, batch=4) …
[PDF->MD] Saved -> output_llm.md

Output: 7,959 characters → output_llm.md


# Checkliste zum Umgang mit externen Dienstleistern

## Einführung

Für das IT-Systemhaus werden im Rahmen von Dienstleistungs- und Werkverträgen unter anderem Freelancer tätig. Der Einsatz erfolgt entweder durch Abrufe aus dem Vertrag „Freelancer“ oder durch Abrufe von Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer. Es kann in der täglichen Arbeit zu Situationen kommen, die als (Teil-) Indizien für das Vorliegen von Scheinselbständigkeit angesehen werden könnten, diese sollten vermieden werden. Solche Indizien liegen insbesondere dann vor, wenn der Verdacht besteht, dass Freelancer in den Arbeitsablauf der BA-IT integriert sind. Die Wertung, ob eine Selbständigkeit oder eine Arbeitnehmereigenschaft vorliegt, richtet sich in jedem Einzelfall nach den Gesamtumständen.

Nachstehende Checkliste dient dem Abgleich, ob in Ihrem Zuständigkeitsbereich in der Zusammenarbeit mit Freelancern Sachverhalte bestehen, die möglicherweise auf das Vorliegen einer Scheinselbständigkeit schließen lassen könnten. Bitte füllen Sie die Liste persönlich unter Beteiligung des Freelancers aus und stimmen Sie das Ergebnis mit Ihrer/Ihrem Vorgesetzten ab. Lfd. Nummern, die mit * gekennzeichnet sind betreffen den Sachverhalt „Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer“.

## Allgemeine Informationen

- Organisationseinheit/ Projekt
- Vertragsname
- Abruf – Nr.
- Name, Vorname (Freelancer)
- Name, Vorname (Abrufverantwortliche/r IT-Systemhaus)
- Name, Vorname (zust. GBL IT-Systemhaus)

<!--page:1-->

## Checkliste

### Lfd. Sachverhalt

1. Für die Kommunikation mit dem Freelancer stehen namentlich benannte Ansprechpartner des IT-Dienstleistungsvermittlers bzw. externen Dienstleistungsunternehmens (im Folgenden: Auftragnehmer) zur Verfügung.  
   **Indizwirkung für Scheinselbständigkeit:** sehr hoch

2. Einsatzzeit und Einsatzumfang des Freelancers wird so gestaltet, dass der Freelancer Aufträge anderer Auftraggeber bearbeiten kann. Der Freelancer ist nicht ausschließlich und auf Dauer für die BA tätig.  
   **Indizwirkung für Scheinselbständigkeit:** sehr hoch

3. Gemeinsame Einsatzpläne werden grundsätzlich nicht aufgestellt, es sei denn, es ist zur Aufgabenerledigung erforderlich und der gemeinsame Einsatzplan ist mit dem namentlich benannten Ansprechpartner des Auftragnehmers abgestimmt. Die ggfs. bestehende Erforderlichkeit ist jeweils begründet und dokumentiert.  
   **Indizwirkung für Scheinselbständigkeit:** sehr hoch

4. Es besteht grundsätzlich keine Anwesenheitspflicht für Freelancer.  
   **Indizwirkung für Scheinselbständigkeit:** sehr hoch

5. Die konkrete Auftragsdefinition und das erwartete Ergebnis werden im Abruf durch die BA definiert und in weiteren konkretisierenden Arbeitsaufträgen beschrieben.  
   **Indizwirkung für Scheinselbständigkeit:** sehr hoch

6. Freelancer nehmen nicht an Dienstbesprechungen teil, es sei denn, dies ist aus fachlichen Gründen zu bestimmten Aspekten oder Tagesordnungspunkten aus Gründen der Auftragserledigung sinnvoll und erforderlich. Die ggfs. bestehende Erforderlichkeit ist im Protokoll jeweils begründet und dokumentiert. Die Teilnehmer werden vor Beginn der Dienstbesprechung über die Anwesenheit Externer unterrichtet.  
   **Indizwirkung für Scheinselbständigkeit:** hoch

7. Freelancer und interne Mitarbeiter sind räumlich getrennt untergebracht, es sei denn die Aufgabenerledigung erfordert die gemeinsame Bearbeitung.  
   **Indizwirkung für Scheinselbständigkeit:** gering

8. Türschilder der Büroräume, in denen Freelancer arbeiten, sind entsprechend beschriftet.  
   **Indizwirkung für Scheinselbständigkeit:** hoch

9. Freelancer besitzen keine BA-Visitenkarten.  
   **Indizwirkung für Scheinselbständigkeit:** hoch

10. Freelancer erhalten PKI-Gastkarten und sind nicht mit „normalen“ Dienstkarten ausgestattet.  
    **Indizwirkung für Scheinselbständigkeit:** hoch

11. Freelancer verfügen nicht über einen BA-spezifischen E-Mail-Account.  
 

## Step 5 — Hybrid Parse: Heuristic + LLM Refinement

Heuristic extracts structure; vLLM fixes heading levels. **Recommended for production** — faster than full LLM, better quality than heuristic alone.


In [37]:
# ── STEP 5: Hybrid parse ─────────────────────────────────────────────────────
md_hybrid = converter.convert(PDF_PATH, output_path="output_hybrid.md", strategy="heuristic+llm")

print(f"\nOutput: {len(md_hybrid):,} characters → output_hybrid.md")
display(Markdown(md_hybrid[:4000]))


[PDF->MD] Input    : Anlage_2_Checkliste_Freelancer_Selbständige .pdf
[PDF->MD] Strategy : heuristic+llm
[PDF->MD] Step 1/2 : Heuristic parsing ...
  Body text size  : 11.0 pt
  Heading sizes   : [14.0]
  HF patterns     : 1 recurring pattern(s) removed
  Content retention : 86.2%  (7,075 / 8,204 body chars)
[PDF->MD] Step 2/2 : LLM refinement pass ...
[PDF->MD] Saved -> output_hybrid.md

Output: 8,225 characters → output_hybrid.md


<!--page:1-->

# Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-

Für das IT-Systemhaus werden im Rahmen von Dienstleistungs- und Werkverträgen unter anderem Freelancer1 tätig. Der Einsatz erfolgt entweder durch Abrufe aus dem Vertrag „Freelancer“ oder durch Abrufe von Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer. Es kann in der täglichen Arbeit zu Situationen kommen, die als (Teil-) Indizien für das Vorliegen von Scheinselbständigkeit angesehen werden könnten, diese sollten vermieden werden. Solche Indizien liegen insbesondere dann vor, wenn der Verdacht besteht, dass Freelancer in den Arbeitsablauf der BA-IT integriert sind. Die Wertung, ob eine Selbständigkeit oder eine Arbeitnehmereigenschaft vorliegt, richtet sich in jedem Einzelfall nach den Gesamtumständen.

Nachstehende Checkliste dient dem Abgleich, ob in Ihrem Zuständigkeitsbereich in der Zusammenarbeit mit Freelancern Sachverhalte bestehen, die möglicherweise auf das Vorliegen einer Scheinselbständigkeit schließen lassen könnten. Bitte füllen Sie die Liste persönlich unter Beteiligung des Freelancers aus und stimmen Sie das Ergebnis mit Ihrer/Ihrem Vorgesetzten ab. Lfd. Nummern, die mit * gekennzeichnet sind betreffen den Sachverhalt „Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer“.

## Organisationseinheit/ Projekt

| Vertragsname |
|---|
| Abruf – Nr. |
| Name, Vorname (Freelancer) |
| Name, Vorname (Abrufverantwortliche/r IT-Systemhaus) |
| Name, Vorname (zust. GBL IT-Systemhaus) |

## Checkliste

| Lfd. Nr. | Sachverhalt | ja | nein | Bemerkungen | Wichtigkeit / Indizwirkung für Scheinselbständigkeit |
|---|---|---|---|---|---|
| 1 * | Für die Kommunikation mit dem Freelancer stehen namentlich benannte Ansprechpartner des IT-Dienstleistungsvermittlers bzw. externen Dienstleistungsunternehmens (im Folgenden: Auftragnehmer) zur Verfügung. |  |  |  | sehr hoch |
| 2 | Einsatzzeit und Einsatzumfang des Freelancers wird so gestaltet, dass der Freelancer Aufträge anderer Auftraggeber bearbeiten kann. Der Freelancer ist nicht ausschließlich und auf Dauer für die BA tätig. |  |  |  | sehr hoch |

### Definition

1 Freelancer = Selbständiger, der aufgrund eines Dienst- oder Werkvertragsverhältnisses Aufträge für den Auftraggeber durchführt.

<!--page:2-->

| Lfd. Nr. | Sachverhalt | ja | nein | Bemerkungen | Wichtigkeit / Indizwirkung für Scheinselbständigkeit |
|---|---|---|---|---|---|
| 3 | Gemeinsame Einsatzpläne werden grundsätzlich nicht aufgestellt, es sei denn, es ist zur Aufgabenerledigung erforderlich und der gemeinsame Einsatzplan ist mit dem namentlich benannten Ansprechpartner des Auftragnehmers abgestimmt. Die ggfs. bestehende Erforderlichkeit ist jeweils begründet und dokumentiert. |  |  |  | sehr hoch |
| 4 | Es besteht grundsätzlich keine Anwesenheitspflicht für Freelancer. |  |  |  | sehr hoch |
| 5 | Die konkrete Auftragsdefinition und das erwartete Ergebnis werden im Abruf durch die BA definiert und in weiteren konkretisierenden Arbeitsaufträgen beschrieben. |  |  |  | sehr hoch |
| 6 | Freelancer nehmen nicht an Dienstbesprechungen teil, es sei denn, dies ist aus fachlichen Gründen zu bestimmten Aspekten oder Tagesordnungspunkten aus Gründen der Auftragserledigung sinnvoll und erforderlich. Die ggfs. bestehende Erforderlichkeit ist im Protokoll jeweils begründet und dokumentiert. Die Teilnehmer werden vor Beginn der Dienstbesprechung über die Anwesenheit Externer unterrichtet. |  |  |  | hoch |
| 7 | Freelancer und interne Mitarbeiter sind räumlich getrennt untergebracht, es sei denn die Aufgabenerledigung erfordert die gemeinsame Bearbeitung. |  |  |  | gering |
| 8 | Türschilder der Büroräume, in denen Freelancer arbeiten, sind entsprechend beschriftet. |  |  |  | hoch |
| 9 | Freelancer besitzen keine BA-Visitenkarten. |  |  |  | hoch |
| 10 | Freelancer erhalten PKI-Gastkarten und sind nicht mit „normalen“ Dienstkarten ausgestattet. |  |  |  | hoch 

## Step 6 — Inspect Heading Hierarchy

Compare the heading outlines from each strategy. Headings should be logically nested with no skipped levels.

In [38]:
# ── STEP 6: Compare heading outlines across strategies ────────────────────────
def heading_outline(md: str, label: str, max_show: int = 30) -> None:
    heading_re = re.compile(r'^(#{1,6})\s+(.+)$', re.MULTILINE)
    headings = heading_re.findall(md)
    level_counts = Counter(len(h) for h, _ in headings)
    print(f"\n{'═'*60}")
    print(f"  {label}  ({len(headings)} headings)")
    print(f"{'─'*60}")
    for lvl in sorted(level_counts):
        print(f"  H{lvl}  ×{level_counts[lvl]}")
    print(f"{'─'*60}")
    for hashes, title in headings[:max_show]:
        indent = "  " * (len(hashes) - 1)
        print(f"  {indent}{hashes} {title}")
    if len(headings) > max_show:
        print(f"  … [{len(headings) - max_show} more]")

# Choose which Markdown to use for all downstream steps
# Options: md_heuristic  |  md_llm  |  md_hybrid
ACTIVE_MD    = md_heuristic     # ← change to md_llm or md_hybrid as needed
ACTIVE_LABEL = "Heuristic"      # label for display

heading_outline(md_heuristic, "Heuristic")

# Uncomment after running steps 4 and 5:
# heading_outline(md_llm,      "LLM (Azure GPT-4o)")
# heading_outline(md_hybrid,   "Hybrid (Heuristic + LLM refine)")


════════════════════════════════════════════════════════════
  Heuristic  (1 headings)
────────────────────────────────────────────────────────────
  H1  ×1
────────────────────────────────────────────────────────────
  # Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-


## Step 7 — Markdown Chunker

Splits the chosen Markdown into context-aware chunks ≤ 2000 characters each.  
Every chunk's `full_text` = `"Heading Path\n\nContent"` — ready for embedding / vector stores.  
`pages` metadata carries which PDF pages the chunk came from.

In [39]:
# ── STEP 7: Chunk the chosen Markdown ────────────────────────────────────────
MAX_CHUNK_SIZE = 2000
DOC_TITLE      = Path(PDF_PATH).stem   # e.g. "annual_report_2024"

chunker = MarkdownChunker(
    max_chunk_size=MAX_CHUNK_SIZE,
    path_separator=" > ",
    doc_title=DOC_TITLE,
)

chunks  = chunker.chunk(ACTIVE_MD)
stats_c = chunker.summary(ACTIVE_MD)

print(f"Strategy      : {ACTIVE_LABEL}")
print(f"Total chunks  : {stats_c['total_chunks']}")
print(f"Unique paths  : {stats_c['unique_paths']}")
print(f"Min / Max     : {stats_c['min_chars']} / {stats_c['max_chars']} chars")
print(f"Avg           : {stats_c['avg_chars']:.0f} chars")
print(f"Total chars   : {stats_c['total_chars']:,}")

Strategy      : Heuristic
Total chunks  : 9
Unique paths  : 1
Min / Max     : 261 / 1958 chars
Avg           : 1049 chars
Total chars   : 9,440


## Step 8 — Validate Chunks

Check size limit compliance, page metadata, and heading breadcrumb paths.

In [40]:
# ── STEP 8a: Size-limit validation ───────────────────────────────────────────
over = [c for c in chunks if c.char_count > MAX_CHUNK_SIZE]
if over:
    print(f"⚠  {len(over)} chunk(s) exceed {MAX_CHUNK_SIZE} chars:")
    for c in over:
        print(f"  Chunk {c.chunk_id}: {c.char_count} chars | {c.heading_path}")
else:
    print(f"✓  All {len(chunks)} chunks are within the {MAX_CHUNK_SIZE}-char limit")

# ── Chunk listing with page range ────────────────────────────────────────────
print(f"\n{'ID':>4}  {'chars':>5}  {'pages':<10}  heading path")
print("─" * 80)
for c in chunks:
    pg = f"p{c.page_start}" if c.page_start == c.page_end else f"p{c.page_start}-{c.page_end}"
    path_display = c.heading_path[:55] + "…" if len(c.heading_path) > 55 else c.heading_path
    print(f"{c.chunk_id:>4}  {c.char_count:>5}  {pg:<10}  {path_display}")

✓  All 9 chunks are within the 2000-char limit

  ID  chars  pages       heading path
────────────────────────────────────────────────────────────────────────────────
   0   1400  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   1    334  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   2    619  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   3    261  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   4   1958  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   5    362  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   6   1777  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   7   1909  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…
   8    820  p1-4        Anlage_2_Checkliste_Freelancer_Selbständige > Checklist…


In [41]:
# ── STEP 8b: Inspect a specific chunk ────────────────────────────────────────
def show_chunk(c: Chunk) -> None:
    bar = "─" * 70
    print(bar)
    print(f"  Chunk ID      : {c.chunk_id}")
    print(f"  Heading path  : {c.heading_path}")
    print(f"  Pages         : {c.pages}  (start={c.page_start}, end={c.page_end})")
    print(f"  Chars         : {c.char_count}")
    print(f"  Hierarchy     : {c.heading_levels}")
    print("  full_text ↓")
    print(bar)
    print(c.full_text[:800])
    if len(c.full_text) > 800:
        print(f"  … [{len(c.full_text) - 800} more chars]")
    print()

# Show first 5 chunks
for c in chunks[:5]:
    show_chunk(c)

──────────────────────────────────────────────────────────────────────
  Chunk ID      : 0
  Heading path  : Anlage_2_Checkliste_Freelancer_Selbständige > Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-
  Pages         : [1, 2, 3, 4]  (start=1, end=4)
  Chars         : 1400
  Hierarchy     : [(1, 'Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-')]
  full_text ↓
──────────────────────────────────────────────────────────────────────
Anlage_2_Checkliste_Freelancer_Selbständige > Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-

Für das IT-Systemhaus werden im Rahmen von Dienstleistungs- und Werkverträgen unter anderem Freelancer1 tätig. Der Einsatz erfolgt entweder durch Abrufe aus dem Vertrag „Free- lancer“ oder durch Abrufe von Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer. Es kann in der täglichen Arbeit zu Situationen kommen, die als (Teil-) Indizien 

In [42]:
# ── STEP 8c: Document outline (unique heading paths) ─────────────────────────
print("Heading breadcrumb outline:\n")
seen: set = set()
for c in chunks:
    if c.heading_path not in seen:
        depth = len(c.heading_levels)
        indent = "  " * max(0, depth - 1)
        pg_tag  = f" [p{c.page_start}]" if c.page_start else ""
        print(f"{indent}{c.heading_path}{pg_tag}")
        seen.add(c.heading_path)

Heading breadcrumb outline:

Anlage_2_Checkliste_Freelancer_Selbständige > Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)- [p1]


In [43]:
# ── STEP 8d: Search chunks by section name ────────────────────────────────────
SEARCH = "Quick Start with Docker"   # e.g. "Introduction"  or  "Methodology"

if SEARCH:
    hits = [c for c in chunks if SEARCH.lower() in c.heading_path.lower()]
    print(f"Found {len(hits)} chunk(s) matching '{SEARCH}':\n")
    for c in hits:
        show_chunk(c)
else:
    print("Set SEARCH to a section keyword to filter chunks.")

Found 0 chunk(s) matching 'Quick Start with Docker':



## Step 9 — Export Chunks to JSON

Each record includes `chunk_id`, `heading_path`, `heading_levels`, `page_start`, `page_end`, `pages`, `content`, `full_text`, `char_count`.  
Feed `full_text` directly to your embedding model.

In [44]:
# ── STEP 9: Export chunks to JSON ────────────────────────────────────────────
CHUNKS_JSON = "chunks.json"
records = chunker.save_json(ACTIVE_MD, CHUNKS_JSON)

# Verify
with open(CHUNKS_JSON, encoding="utf-8") as f:
    saved = json.load(f)

print(f"\nVerified: {len(saved)} chunks in '{CHUNKS_JSON}'")
print(f"Record fields: {list(saved[0].keys())}")
print("\nSample record:")
sample = {k: v for k, v in saved[0].items() if k != "full_text"}  # exclude long field
sample["full_text"] = saved[0]["full_text"][:200] + "…"
print(json.dumps(sample, indent=2, ensure_ascii=False))

[Chunker] Saved 9 chunks → chunks.json

Verified: 9 chunks in 'chunks.json'
Record fields: ['chunk_id', 'heading_path', 'heading_levels', 'page_start', 'page_end', 'pages', 'metadata', 'content', 'full_text', 'char_count']

Sample record:
{
  "chunk_id": 0,
  "heading_path": "Anlage_2_Checkliste_Freelancer_Selbständige > Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-",
  "heading_levels": [
    [
      1,
      "Checkliste zum Umgang mit externen Dienstleistern -hier Freelancer (Selbständige)-"
    ]
  ],
  "page_start": 1,
  "page_end": 4,
  "pages": [
    1,
    2,
    3,
    4
  ],
  "metadata": {},
  "content": "Für das IT-Systemhaus werden im Rahmen von Dienstleistungs- und Werkverträgen unter anderem Freelancer1 tätig. Der Einsatz erfolgt entweder durch Abrufe aus dem Vertrag „Free- lancer“ oder durch Abrufe von Freelancern aus anderen Dienstleistungs- und Werkverträgen als Subunternehmer. Es kann in der täglichen Arbeit zu Situationen kommen, 

## Step 3 — Content-Integrity Check

Reviews what was removed (headers/footers) and what was retained.  
`content_retention_pct` close to 100% is healthy. Low values → inspect `hf_dropped_samples`.

## Test — Unified HTML → Markdown → Chunk Pipeline

Tests the full path: `HTML2MarkdownParser.load_data()` → `_chunk_markdown_documents()` → `TextNode` list.  
Validates that heading breadcrumbs, table splitting, and metadata propagation all work correctly.

In [45]:
# ── Reload modules ────────────────────────────────────────────────────────────
import importlib
import html_to_markdown as htm
import chunking as chk
import markdown_chunker as mc

importlib.reload(mc)
importlib.reload(htm)
importlib.reload(chk)

from html_to_markdown import HTML2MarkdownParser
from chunking import _chunk_markdown_documents

print("Modules reloaded OK")

Modules reloaded OK


In [46]:
# ── Step A: Parse HTML → LlamaIndex Document ─────────────────────────────────
from pathlib import Path

parser = HTML2MarkdownParser()
docs = parser.load_data(Path("test_sample.html"))

print(f"Documents returned: {len(docs)}")
print(f"Metadata: {docs[0].metadata}")
print(f"Markdown length: {len(docs[0].text):,} chars")
print("\n── First 800 chars of Markdown ──")
print(docs[0].text[:800])

-> Parsing HTML file: test_sample.html
   Detected encoding: utf-8
Documents returned: 1
Metadata: {'title': 'Quarterly Financial Report Q4 2025'}
Markdown length: 2,400 chars

── First 800 chars of Markdown ──
# Quarterly Financial Report Q4 2025

## Executive Summary

This report presents the financial performance for the fourth quarter of fiscal year 2025. Overall revenue grew by 12% year-over-year, driven primarily by strong performance in the Digital Services and Cloud Infrastructure segments. 

## Revenue Breakdown

### By Segment

Segment | Q4 2025 ($M) | Q4 2024 ($M) | YoY Change  
---|---|---|---  
Digital Services | 4,250 | 3,680 | +15.5%  
Cloud Infrastructure | 3,120 | 2,810 | +11.0%  
Consulting | 2,890 | 2,750 | +5.1%  
Legacy Systems | 1,340 | 1,520 | -11.8%  

### By Region

North America contributed 58% of total revenue, followed by EMEA at 27% and APAC at 15%. The APAC region showed the strongest growth at 18% year-over-year. 

## Key Metrics

### Profitability

* **G

In [47]:
# ── Step B: Chunk the documents ───────────────────────────────────────────────
nodes = _chunk_markdown_documents(docs, max_chunk_size=500)

print(f"Total nodes: {len(nodes)}\n")
for i, node in enumerate(nodes):
    pages_info = ""
    if node.metadata.get("pages"):
        pages_info = f"  pages={node.metadata['pages']}"
    print(f"  Node {i}: {len(node.text):>4} chars | "
          f"path={node.metadata.get('heading_path', '(none)')}{pages_info}")

Total nodes: 9

  Node 0:  325 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Executive Summary
  Node 1:  345 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Revenue Breakdown > By Segment
  Node 2:  260 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Revenue Breakdown > By Region
  Node 3:  232 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Key Metrics > Profitability
  Node 4:  251 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Key Metrics > Cash Flow
  Node 5:  469 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Strategic Initiatives > AI and Automation
  Node 6:  326 chars | path=Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Strategic Initiatives > Sustainability
  Node 7:  394 chars | path=Quarterly Financial Report Q4 

In [48]:
# ── Step C: Inspect sample nodes ──────────────────────────────────────────────
from IPython.display import display, Markdown
import json

# Show first node's full text (rendered as markdown) 
print("═" * 60)
print("NODE 0 — full_text (what the embedding model sees):")
print("═" * 60)
display(Markdown(nodes[0].text))

print("\n" + "═" * 60)
print("NODE 0 — metadata:")
print("═" * 60)
print(json.dumps(nodes[0].metadata, indent=2, ensure_ascii=False))

════════════════════════════════════════════════════════════
NODE 0 — full_text (what the embedding model sees):
════════════════════════════════════════════════════════════


Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Executive Summary

This report presents the financial performance for the fourth quarter of fiscal year 2025. Overall revenue grew by 12% year-over-year, driven primarily by strong performance in the Digital Services and Cloud Infrastructure segments.


════════════════════════════════════════════════════════════
NODE 0 — metadata:
════════════════════════════════════════════════════════════
{
  "title": "Quarterly Financial Report Q4 2025",
  "heading_path": "Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Executive Summary"
}


In [49]:
# ── Step D: Validate chunk properties ─────────────────────────────────────────
sizes = [len(n.text) for n in nodes]

print(f"Chunk count     : {len(nodes)}")
print(f"Min size        : {min(sizes):,} chars")
print(f"Max size        : {max(sizes):,} chars")
print(f"Avg size        : {sum(sizes) / len(sizes):,.0f} chars")
print(f"Total chars     : {sum(sizes):,}")

# Verify no page metadata on HTML (only PDFs should have it)
pages_present = [n for n in nodes if n.metadata.get("pages")]
print(f"\nNodes with page metadata: {len(pages_present)} (expected: 0 for HTML)")

# Verify heading paths are populated
paths = {n.metadata.get("heading_path", "") for n in nodes}
print(f"Unique heading paths: {len(paths)}")
for p in sorted(paths):
    print(f"  • {p}")

# Verify all nodes have the HTML title in metadata
titles = {n.metadata.get("title", "") for n in nodes}
print(f"\nDocument title(s): {titles}")

# Check that the table node contains pipe syntax
table_nodes = [n for n in nodes if "|" in n.text and "---" in n.text]
print(f"Nodes containing tables: {len(table_nodes)}")
if table_nodes:
    print("\n── Table node preview ──")
    print(table_nodes[0].text[:500])

Chunk count     : 9
Min size        : 232 chars
Max size        : 469 chars
Avg size        : 336 chars
Total chars     : 3,023

Nodes with page metadata: 0 (expected: 0 for HTML)
Unique heading paths: 9
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Executive Summary
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Key Metrics > Cash Flow
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Key Metrics > Profitability
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Outlook
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Revenue Breakdown > By Region
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Revenue Breakdown > By Segment
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Risk Factors
  • Quarterly Financial Report Q4 2025 > Quarterly Financial Report Q4 2025 > Strategic Initiative